# Deep Learning with PyTorch · Dataset and DataLoader

Macademia · For adults

> **Running in Colab:** Open this file at [colab.research.google.com](https://colab.research.google.com) with **File → Upload notebook**. For a GPU choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom (Shift + Enter). PyTorch is preinstalled in Colab; to run on your own computer use `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # reproducibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Your own Dataset class

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
class SensorVerisi(Dataset):
    def __init__(self, n=1000):
        g = torch.Generator().manual_seed(0)
        self.X = torch.randn(n, 3, generator=g); self.y = (self.X.sum(1) > 0).long()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]
ds = SensorVerisi()
print(len(ds), ds[0])

## 2. Split and batches

In [ ]:
tr, va = random_split(ds, [800, 200], generator=torch.Generator().manual_seed(42))
for bs, drop in [(64, False), (64, True), (100, False)]:
    dl = DataLoader(tr, batch_size=bs, shuffle=True, drop_last=drop)
    sizes = [len(xb) for xb, _ in dl]
    print(f'bs={bs} drop_last={drop}: len={len(dl)} last={sizes[-1]}')                # last batch size

## 3. Is shuffling reproducible?

In [ ]:
def first_batch(seed):
    dl = DataLoader(range(20), batch_size=5, shuffle=True, generator=torch.Generator().manual_seed(seed))
    return next(iter(dl)).tolist()
print(first_batch(1), first_batch(1), first_batch(2))

## 4. Using it in a training loop

In [ ]:
from torch import nn
model = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
tr_dl = DataLoader(tr, batch_size=64, shuffle=True); va_dl = DataLoader(va, batch_size=256)
for epoch in range(5):
    model.train()
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); nn.functional.cross_entropy(model(xb), yb).backward(); opt.step()
    model.eval(); ok = 0
    with torch.no_grad():
        for xb, yb in va_dl:
            ok += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    print(epoch, ok / len(va))

## Exercises
1. Write a Dataset that reads a CSV file.
2. Try `num_workers=2` and measure the time.
3. Train on class-sorted data with `shuffle=False`; compare the result.